# Raw data audit example

This notebook imports the audit implementation from `src/job_fraud/`. It reads the raw data without modifying it. No cleaning, embedding generation, or training occurs.

Use the `job-fraud` environment. Notebook execution tooling is not installed yet.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the project explicitly from the launch directory.
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / 'environment.yml').is_file()
                    and (path / 'src/job_fraud').is_dir())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from job_fraud.data import TEXT_FIELDS, load_postings
from job_fraud.audit import build_audit

In [ ]:
postings = load_postings(PROJECT_ROOT / 'data/raw/fake_job_postings.csv')
report, duplicate_memberships = build_audit(postings)
postings[['job_id', 'title', 'location', 'fraudulent']].head()

## Labels and missing fields
`0` means labeled legitimate; `1` means labeled fraudulent. Missingness and label counts come from the shared audit code.

In [ ]:
pd.Series(report['label_counts'], name='posting_count').rename_axis('fraudulent')

In [ ]:
pd.DataFrame(report['missingness']).T[['empty_count', 'empty_percent']]

## Text lengths
These are whitespace word counts, not BERT/OpenAI tokens. Raw HTML is retained. Combined text is a measurement, not finalized model input.

In [ ]:
pd.DataFrame({field: stats['whitespace_words'] for field, stats in report['text_lengths'].items()}).T

## Duplicate candidates and label conflicts
Groups ignore posting IDs and labels when matching. Normalized text matching only ignores case and repeated whitespace. It does not detect all near duplicates. Counts overlap across definitions; do not add them.

Review conflicting labels and related postings before assigning shared splits.

In [ ]:
pd.DataFrame(report['duplicates']).T[['groups', 'postings_in_groups', 'groups_with_conflicting_labels']]

In [ ]:
duplicate_memberships.loc[duplicate_memberships['label_conflict']].head(10)

## Save an audit report
From the project root, run `PYTHONPATH=src mamba run -n job-fraud python -m job_fraud.audit`. Reports are written to a new timestamped folder under `reports/data_audit/`, which is excluded from Git.